# 02 EDA

How Severity changes with the weather, time and road features the models use. Only the train split is used here, so nothing from val/test leaks into model choices.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

sys.path.insert(0, str(Path.cwd().parent))  # so src imports from the notebooks folder
from src import config

config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# blues for severity 1 to 4, light to dark
SEVERITY_COLORS = ['#86b6ef', '#3987e5', '#1c5cab', '#0d366b']
BLUE = '#2a78d6'
ORANGE = '#eb6834'
INK = '#52514e'
plt.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': INK,
    'xtick.color': INK, 'ytick.color': INK, 'text.color': '#1a1a19',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e1e0d9', 'grid.linewidth': 0.8,
    'axes.axisbelow': True, 'font.size': 10,
})

In [ ]:
# share of each severity within a group, as a 100% stacked bar
def plot_severity_share(table, title, filename, figsize=(8, 4)):
    share = table.div(table.sum(axis=1), axis=0) * 100
    fig, ax = plt.subplots(figsize=figsize)
    bottom = pd.Series(0.0, index=share.index)
    for sev, color in zip(config.CLASSES, SEVERITY_COLORS):
        ax.bar(share.index.astype(str), share[sev], bottom=bottom, color=color,
               edgecolor='#fcfcfb', linewidth=1, label=f'Severity {sev}')
        bottom += share[sev]
    ax.set_ylabel('% of accidents')
    ax.set_ylim(0, 100)
    ax.set_title(title, loc='left', fontweight='bold')
    ax.grid(axis='x', visible=False)
    ax.legend(ncol=4, loc='upper left', bbox_to_anchor=(0, -0.12), frameon=False)
    fig.tight_layout()
    fig.savefig(config.FIGURES_DIR / filename, dpi=150)
    plt.show()
    return share.round(1)

## Load

In [ ]:
df = pd.read_parquet(config.TRAIN_PARQUET)
print(f'{len(df):,} train rows, {df.shape[1]} columns')

# smaller sample for the slow plots
sample = df.sample(500_000, random_state=config.RANDOM_SEED)

## Weather

Severity mix for each weather group (the ~140 raw labels were grouped in `clean()`).

In [ ]:
by_weather = pd.crosstab(df['weather_group'], df[config.TARGET])
by_weather = by_weather.loc[by_weather.sum(axis=1).sort_values(ascending=False).index]
by_weather.assign(total=by_weather.sum(axis=1))

In [ ]:
plot_severity_share(by_weather, 'Severity mix by weather group', 'severity_by_weather.png', figsize=(10, 4))

## Time of day and day of week

In [ ]:
by_hour = pd.crosstab(df['hour'], df[config.TARGET])
plot_severity_share(by_hour, 'Severity mix by hour of day', 'severity_by_hour.png', figsize=(10, 4))

In [ ]:
DAYS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
by_day = pd.crosstab(df['day_of_week'], df[config.TARGET])
by_day.index = DAYS
plot_severity_share(by_day, 'Severity mix by day of week', 'severity_by_weekday.png')

## Road features

For each road feature: what % of accidents are severity 3 or 4 when the feature is nearby vs not, and how common the feature is.

In [ ]:
severe = df[config.TARGET] >= 3
rows = []
for col in config.ROAD_COLS:
    present = df[col] == 1
    rows.append({
        'feature': col,
        'present_%': present.mean() * 100,
        'severe_if_present_%': severe[present].mean() * 100,
        'severe_if_absent_%': severe[~present].mean() * 100,
    })
road = pd.DataFrame(rows).set_index('feature').sort_values('severe_if_present_%')
road.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
y = np.arange(len(road))
h = 0.38
ax.barh(y + h / 2, road['severe_if_present_%'], height=h, color=BLUE, label='feature nearby')
ax.barh(y - h / 2, road['severe_if_absent_%'], height=h, color=ORANGE, label='not nearby')
ax.set_yticks(y, road.index)
ax.axvline(severe.mean() * 100, color=INK, linewidth=1, linestyle='--')
ax.text(severe.mean() * 100 + 0.3, len(road) - 0.4, 'overall', color=INK, fontsize=8.5)
ax.set_xlabel('% of accidents that are severity 3 or 4')
ax.set_title('Severe accidents near each road feature', loc='left', fontweight='bold')
ax.grid(axis='y', visible=False)
ax.legend(loc='lower right', frameon=False)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / 'road_features_severity.png', dpi=150)
plt.show()

## Numeric weather by severity

Box plots on a 500k-row sample (outlier dots hidden). Missing values are left out here; the models impute them.
Visibility is not plotted because almost every value is exactly 10 mi (the reporting cap), so it gets its own table below.

In [ ]:
cols = ['Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Wind_Speed(mph)']
fig, axes = plt.subplots(1, len(cols), figsize=(12, 3.8))
for ax, col in zip(axes, cols):
    groups = [sample.loc[sample[config.TARGET] == s, col].dropna() for s in config.CLASSES]
    box = ax.boxplot(groups, tick_labels=config.CLASSES, showfliers=False, patch_artist=True,
                     medianprops={'color': '#fcfcfb', 'linewidth': 1.5})
    for patch, color in zip(box['boxes'], SEVERITY_COLORS):
        patch.set_facecolor(color)
        patch.set_edgecolor(color)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('Severity')
    ax.grid(axis='x', visible=False)
fig.suptitle('Numeric weather by severity', x=0.01, ha='left', fontweight='bold')
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / 'weather_numeric_by_severity.png', dpi=150)
plt.show()

In [ ]:
sample.groupby(config.TARGET)[cols].median()

In [ ]:
# % under 10 mi visibility and % with any rain, per severity
pd.DataFrame({
    'visibility_under_10mi_%': (sample['Visibility(mi)'] < 10).groupby(sample[config.TARGET]).mean() * 100,
    'any_precip_%': (sample['Precipitation(in)'] > 0).groupby(sample[config.TARGET]).mean() * 100,
}).round(1)

## Correlations

Pearson correlation between the numeric and flag features and Severity (500k sample). Blue = positive, red = negative. Blank cells (like Temperature vs temp_missing) can't be computed, because the flag is always 0 wherever the value exists.

In [ ]:
corr_cols = (config.NUMERIC_WEATHER + list(config.MISSING_FLAGS.values())
             + ['is_night', 'road_feature_count', 'hour', 'is_rush_hour', 'is_weekend', config.TARGET])
corr = sample[corr_cols].astype(float).corr()
diverging = LinearSegmentedColormap.from_list('div', ['#e34948', '#f0efec', '#1c5cab'])

fig, ax = plt.subplots(figsize=(10, 8.5))
im = ax.imshow(corr, cmap=diverging, vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=60, ha='right')
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        v = corr.iloc[i, j]
        if abs(v) >= 0.1 and i != j:
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7,
                    color='#fcfcfb' if abs(v) > 0.6 else '#1a1a19')
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.7)
ax.set_title('Feature correlations (|r| >= 0.1 labelled)', loc='left', fontweight='bold')
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / 'feature_correlation.png', dpi=150)
plt.show()

In [ ]:
# correlation of each feature with severity, strongest first
corr[config.TARGET].drop(config.TARGET).sort_values(key=abs, ascending=False).round(3)

## Year and Source

Notebook 01 showed the severity labels shift by year and by Source. This table shows how the two overlap, to help decide if `year` or `Source` should be model features.

In [ ]:
pd.crosstab(df['year'], df['Source'], margins=True)

In [ ]:
# severity mix for each year and source
mix = pd.crosstab([df['year'], df['Source']], df[config.TARGET], normalize='index') * 100
mix.round(1)

## Observations

- Weather: a weak signal, only a few points of difference between groups.
- Time: late night and weekends are more severe; midnight to 3am has about 5–6% severity 4 vs 2% in the day.
- Road features: the clearest pattern. Junctions are 28% severity 3 or 4; stops, crossings and signals 2–9%. My guess is these are slow city streets, while junctions are on faster roads.
- Numeric weather: about the same for every severity, except temperature (severity 1 is warmer, median 74°F vs 61–64°F). Visibility is almost always 10 mi.
- Correlations: nothing strong (max r = 0.18). The weather columns go missing together, and precip_missing is probably standing in for Source and year.
- Year and Source: the labels change over time, for example Source1 has 0% severity 3 from 2021 on.
- What this means for the models: no single feature is strong, so I expect modest macro-F1. Road and time should help more than weather. Year and Source stay out of the main models since they mostly capture the labelling change; I'll test one Decision Tree with them added.